In [ ]:
# 함수로 스레드 실행
import threading
import time

def greeting(name:str) -> None:
    print(f"[{name}] 시작")
    time.sleep(2)       # 2초 간 wait. lock 해제.
    print(f"[{name}] 완료")

# 1) 함수 이용하여 스레드 생성: (target = 실행할 함수 지정, args=인자값)
# 1. 철수의 인사하기 스레드 생성
t1 = threading.Thread(target=greeting, args=("철수",))  # args는 튜플이어야.

# 2. 영희의 인사하기 스레드 생성
t2 = threading.Thread(target=greeting, args=("영희",))

# 프로세스 안에 메인 스레드 실행. t1 스레드 생성, t2 스레드 생성

# 2) 스레드 시작(start())
t1.start()

# "[철수] 시작" 출력. 2초 wait.(Lock 던지면 main thread로 넘어옴.)

t2.start()

# main thread에서 Lock 받아와 "[영희] 시작" 출력. 2초 wait -> main thread

# print("모든 작업 완료")

# main thread에서 "모든 작업 완료" 출력 후 main thread 종료
# but t1, t2 thread는 살아있음.
# "[철수] 완료" 출력 후 t1 thread 종료
# "[영희] 완료" 출력 후 t2 thread 종료
# 모든 thread 종료되었을 때 프로세스 종료


# 첫 코드 출력
# [철수] 시작
# [영희] 시작
# 모든 작업 완료
# [철수] 완료[영희] 완료
# 위 결과는 논리적으로 말 안 됨.(main thread 먼저 끝나고 t1,t2 thread 끝남.)

# 3) 두 스레드가 끝날 때까지 메인 스레드를 기다리게 만들어야 한다.
t1.join()       # 메인 스레드에 대한 접근이 안 됨. t1, t2를 조작해야 함.
t2.join()       # t2.join(): t2 끝날 때까지 현재 thread wait

print("모든 작업 완료")

[철수] 시작
[영희] 시작
[철수] 완료
[영희] 완료
모든 작업 완료


In [1]:
# 클래스로 스레드 생성(상태 저장이 필요할 때 사용. 함수로는 상태 저장이 안 되니까)
import threading
import time

# Thread 상속받은 클래스는 thread가 된다.
class download(threading.Thread):
    def __init__(self, url:str):
        super().__init__()      # Thread 클래스의 __init__() 필수 호출
        self.url = url
        self.state = None       # 초기라서 None. thread가 자기 상태 저장
    # t1.start() : thread object 안에 있는 run() 호출
    # 스레드 처리해야 하는 클래스 반드시 run() 메소드 존재
    # run() 메소드 안에 있는 코드가 스레드로 작동
    def run(self) -> None:
        print(f"다운로드 시작 : {self.url}")    # 다운로드할 때 기다리는 동안 다른 작업
        time.sleep(1)
        self.state = f"{self.url}의 데이터 100kb"


# 다운로드 하는 3개 스레드 생성
ops = [download(f"https://api.aaa.com/{i}") for i in range(3)]

# 스레드 3개 start()
for op in ops:
    op.start()
# 스레드 3개 join()
for op in ops:
    op.join()
# state 출력
for op in ops:
    print(f"결과 : {op.state}")

print("다운로드 종료")

다운로드 시작 : https://api.aaa.com/0
다운로드 시작 : https://api.aaa.com/1
다운로드 시작 : https://api.aaa.com/2
결과 : https://api.aaa.com/0의 데이터 100kb
결과 : https://api.aaa.com/1의 데이터 100kb
결과 : https://api.aaa.com/2의 데이터 100kb
다운로드 종료


In [ ]:
# 일반 스레드: 메인이 끝나도 자기 일이 끝날 때까지 끝나지 않는 스레드

# 데몬 스레드: 메인이 끝나면 강제로 함께 종료되는 스레드. 로깅, 모니터링에 적합

# 클래스로 스레드 이용 시 데몬스레드 설정(위 클래스 복사)
class download(threading.Thread):
    def __init__(self, url:str):
        super().__init__()
        self.url = url
        self.state = None
        # self.daemon = True      # True이면 데몬 스레드가 된다.

    def run(self) -> None:
        print(f"다운로드 시작 : {self.url}")
        time.sleep(1)
        self.state = f"{self.url}의 데이터 100kb"


# 함수로 스레드 이용 시 데몬스레드 설정(위 함수 복사)
t2 = threading.Thread(target=greeting, args=("영희",))

In [13]:
# 동기화: 공유 변수를 관리하는 것.
# 문제 상황 : 경쟁 상태(Race Condition)
# 스레드는 메모리를 공유하므로, 같은 변수를 동시에 건드리면 값이 깨짐.
# 1개의 은행 계좌를 2명이 동시에 입금하려 함.
import threading
import time

balance = 0     # global var : 공유 변수

# 스레드로 작동될 함수
def deposit():
    global balance
    for _ in range(50_000):     # _로 단위 구분 가능
        now = balance           # 현재 잔액 가져오기
        time.sleep(0)           
        balance = now + 1       # 1원씩 입금

t1 = threading.Thread(target=deposit)   # 스레드 생성
t2 = threading.Thread(target=deposit)
# ts = [threading.Thread(target=deposit) for _ in range(2)]

t1.start()
t2.start()
t1.join()
t2.join()

# t1의 balance 0인 상태에서 wait, t2의 balance 0 넣고 now 0인 상태에서 wait
# t1 다시 돌아가 실행(balance=1, now=1), t2의 now는 여전히 0. 동기화되지 않음.

print(f"기대값: {100_000}")
print(f"실제값: {balance}")

# 기대값: 100000
# 실제값: 50372
# 동기화가 안 됨.

기대값: 100000
실제값: 50372


In [15]:
import threading
import time

balance = 0
# Lock 생성
lock = threading.Lock()     # 한 번에 한 스레드만 진입, 재진입 불가. for문 없으면 한 번 돌고 끝.
# RLock() : 재진입 가능한 자물쇠
# Semaphore : 개수 제한하는 자물쇠     # "몇 번째 대기 중입니다."
# Event : 신호 보내는 자물쇠 -> 한 스레드가 "준비완료" 신호를 주고 다른 스레드가 기다림
# Condition : 조건 대기 -> 특정 조건이 될 때까지 대기/통지
# Barrier : 집결 지정 -> N개 스레드가 모두 도착할 때까지 대기(예. YouTube(영상, 소리 다 들어와야 진행 가능))

# 스레드로 작동될 함수
def deposit():
    global balance
    for _ in range(50_000):
        with lock:          # lock을 걸어 블록이 실행될 때까지는 다른 스레드에서 접근x
            now = balance
            time.sleep(0)       # t1 lock 중이면 time.sleep()이어도 t2 접근 불가
            balance = now + 1 

t1 = threading.Thread(target=deposit)
t2 = threading.Thread(target=deposit)
# ts = [threading.Thread(target=deposit) for _ in range(2)]

t1.start()
t2.start()
t1.join()
t2.join()

print(f"기대값: {100_000}")
print(f"실제값: {balance}")


기대값: 100000
실제값: 100000


In [20]:
# Semaphore: 서버 과부하로 인한 차단 방지에 필수(서버가 1000명 한도라면 1000명까지는 접속 허용, 나머지는 대기)
# 현재 서버의 용량은 동시 접속 3명까지. 10명이 접속.
import threading
import time

accessRestriction=threading.Semaphore(3)     # 동시접속 3명까지 허용

def regist(number: int):
    with accessRestriction:
        print(f"{number} 글카 요청 시작")
        time.sleep(1)   # 1년 사용했다 가정
        print(f"{number} 글카 반납")


threads = [ threading.Thread(target=regist, args=(i,)) for i in range(10)]

for t in threads:
    t.start()

for t in threads:
    t.join()

print("글카 대여 사업 종료")

0 글카 요청 시작
1 글카 요청 시작
2 글카 요청 시작
0 글카 반납
1 글카 반납
3 글카 요청 시작
4 글카 요청 시작
2 글카 반납
5 글카 요청 시작
3 글카 반납
5 글카 반납
6 글카 요청 시작
7 글카 요청 시작
4 글카 반납
8 글카 요청 시작
8 글카 반납
9 글카 요청 시작
7 글카 반납
6 글카 반납
9 글카 반납
글카 대여 사업 종료


In [ ]:
# 스레드 다루는 고수준(기능이 많다. 간단하게 사용할 수 있다.) API : ThreadPoolExecutor(3.2부터) 
# 상세 제어는 어렵. 하지만 빠름. 순서 보장
# 기본 사용법
# 주식 가격 5종목

from concurrent.futures import ThreadPoolExecutor
import time

def get_price(stock:str)-> dict:
    time.sleep(1)       # sleep이 있어야 다른 종목도 일함.
    return {"종목":stock, "가격":3000000}

stocks = ["삼성전자", "하이닉스", "네이버", "카카오", "현대차"]

with ThreadPoolExecutor(max_workers=5) as pools:        # 종목 개수만큼 max_workers 설정. 200 종목이면 200
    results = pools.map(get_price, stocks)           # map: 입력 순서대로 결과 반환

for result in results:
    print(result)                                   # 리스트 순서 보장. 빠름(5개 스레드 사용해서 1초 걸림)

{'종목': '삼성전자', '가격': 3000000}
{'종목': '하이닉스', '가격': 3000000}
{'종목': '네이버', '가격': 3000000}
{'종목': '카카오', '가격': 3000000}
{'종목': '현대차', '가격': 3000000}


In [30]:
# 먼저 끝난 것부터 처리: 비동기
from concurrent.futures import ThreadPoolExecutor, as_completed
import time, random

def task(number:int)->str:
    duration = random.uniform(0.5, 0.2)
    time.sleep(duration)        # 임의의 시간 만큼 wait
    return f"작업 {number} ({duration: .1f})"

with ThreadPoolExecutor(max_workers=4) as pools:        # max_workers=cpu core 수에 맞춰서 설정
    futures = [pools.submit(task,i) for i in range(6)]      
    # 스레드는 4개, task는 6개. submit은 만들어 줄래? 요청한 것. 스레드 다 차있으면 대기
    for future in as_completed(futures):    # 끝나는 순서대로 수신
        try:
            print(f"완료: {future.result()}")
        except Exception as e:
            print(f"실패: {e}")

완료: 작업 0 ( 0.2)
완료: 작업 2 ( 0.3)
완료: 작업 3 ( 0.3)
완료: 작업 1 ( 0.5)
완료: 작업 5 ( 0.2)
완료: 작업 4 ( 0.4)


In [ ]:
# 많이 쓰임.
# Thread 이용해서 데이터 주고 받기.(kafka가 이 구조?)
# 주고 받는 타이밍이 맞아야?
# 생산자-소비자 패턴: Queue 구조 활용(FIFO)

# Producer - Consumer
import threading
import queue
import time

workingQueue = queue.Queue()    # 큐 생성, 큐: FIFO 저장소

# 예. 손님은 빵을 기다리고 있다. 제빵사가 빵을 만들어 선반(Queue)에 올리면 손님은 빵을 가져간다.(FIFO로)
def producer():
    for i in range(5):
        workingQueue.put(f"데이터-{i}")
        print(f"생산: 데이터-{i}")
        time.sleep(0.3)
    workingQueue.put(None)      # 데이터 없는 것 표시하기 위함


def consumer():
    while True:
        item = workingQueue.get()   # get() 했을 때 값이 있을지 없을지 모름.
        if item is None:
            workingQueue.task_done()    # task_done: 한 번의 일을 끝냈다.
            break
        print(f"소비 : {item} 처리 완료")
        workingQueue.task_done()        # 한 번의 일을 끝냈다.

In [35]:
t1 = threading.Thread(target=producer)
t2 = threading.Thread(target=consumer)

t1.start()
t2.start()
t1.join()
t2.join()

print("프로그램 종료")
# 순서대로 소비. 나오는 대로 바로 소비.

생산: 데이터-0
소비 : 데이터-0 처리 완료
생산: 데이터-1소비 : 데이터-1 처리 완료

생산: 데이터-2소비 : 데이터-2 처리 완료

생산: 데이터-3소비 : 데이터-3 처리 완료

생산: 데이터-4소비 : 데이터-4 처리 완료

프로그램 종료
